# Конспект: Цепи Маркова, HMM и алгоритм Витерби


# 1. Цепи Маркова

Цепь Маркова — это модель случайного процесса, где будущее зависит только от текущего состояния.

Не от всей истории.

![](markov.png)


## Марковское свойство

$$
P(X_{t+1}=j \mid X_t=i, X_{t-1}, ..., X_1)
=
P(X_{t+1}=j \mid X_t=i)
$$

Чтобы предсказать следующий шаг, нам достаточно знать текущее состояние.


## Компоненты модели

1) Множество состояний $Q$  
2) Матрица переходов $A$:

$$
A_{i,j} = P(X_{t+1}=j \mid X_t=i)
$$

Каждая строка — распределение вероятностей.

3) Начальное распределение $\pi$:

$$
\pi_j = P(X_1=j)
$$


## Вероятность последовательности

Для траектории $x_1, ..., x_T$:

$$
P(x_{1:T}) =
\pi_{x_1}
\prod_{t=1}^{T-1} A_{x_t, x_{t+1}}
$$


## Почему используем логарифмы

Произведение многих вероятностей даёт очень маленькое число (underflow).

Используем:

$$
\log(a \cdot b) = \log a + \log b
$$

Тогда:

$$
\log P =
\log \pi +
\sum_{t=1}^{T-1} \log A_{x_t,x_{t+1}}
$$

Преимущества:

- численная стабильность
- проще искать максимум


# 2. Hidden Markov Model (HMM)

это расширение цепи Маркова, где:
- состояния существуют,
- но мы их не наблюдаем напрямую.

Мы наблюдаем только результаты их «работы».

Разница между обычной цепью Маркова и HMM

🔹 Обычная цепь Маркова
Мы видим сами состояния:
* Rain → Rain → Sun → Sun<br>
Состояния = наблюдения.

🔹 HMM
Есть два уровня:
* Скрытые состояния, которые подчиняются цепи Маркова:   NN → VB → NN, 
* Наблюдения (слова):                                    dogs  eat  pizza

Мы видим слова, но не видим теги.

Теги скрыты → поэтому модель называется Hidden.

![](hmm.png)

## Структура HMM

1) Начальное распределение:

$$
\pi_i = P(y_1=i)
$$

2) Матрица переходов (как в цепи Маркова):

$$
A_{i,j} = P(y_{t+1}=j \mid y_t=i)
$$

3) Матрица эмиссий (вероятность того, что состояние j «сгенерирует» наблюдение x):

$$
B_j(x) = P(x_t=x \mid y_t=j)
$$


Процесс работы можно описать так:
* Сначала система выбирает скрытое состояние.
* Потом это состояние «производит» наблюдение.
* Потом скрытое состояние меняется по правилам цепи Маркова.
* И снова генерируется наблюдение.

## Полная формула HMM
Совместная вероятность

$$
P(x_{1:T}, y_{1:T}) =
\pi_{y_1}
\prod_{t=2}^{T} A_{y_{t-1},y_t}
\prod_{t=1}^{T} B_{y_t}(x_t)
$$

Это:

- цепь Маркова по скрытым состояниям
- плюс генерация наблюдений


# 3. Алгоритм Витерби
Алгоритм Витерби — это способ найти самую вероятную последовательность скрытых состояний в модели HMM.

То есть мы видим наблюдения (например, слова), и хотим понять — какие скрытые состояния (например, POS-теги) были наиболее вероятны.

Задача:

$$
y^* = \arg\max_{y_{1:T}} P(y_{1:T} \mid x_{1:T})
$$

Найти самую вероятную последовательность скрытых состояний.


## Почему нельзя перебрать всё

Если 10 тегов и 20 слов:

$$
10^{20}
$$

вариантов.

Перебор невозможен.


## Идея

Используем динамическое программирование:

- Идём слева направо.
- На каждом шаге храним только лучший путь, который приводит к каждому состоянию.
- В конце восстанавливаем лучший глобальный путь.

### Инициализация

$$
\delta_1(j)=\pi_j \, B_j(x_1),\qquad \psi_1(j)=0
$$

### Рекурсия (для \(t=2,\dots,T\))

$$
\delta_t(j)=\left(\max_{i\in\{1,\dots,N\}} \delta_{t-1}(i)\,A_{i,j}\right)\,B_j(x_t)
$$

$$
\psi_t(j)=\arg\max_{i\in\{1,\dots,N\}} \delta_{t-1}(i)\,A_{i,j}
$$

### Завершение

$$
P^*=\max_j \delta_T(j), \qquad y_T^*=\arg\max_j \delta_T(j)
$$

### Восстановление пути (назад)

$$
y_{t}^*=\psi_{t+1}(y_{t+1}^*), \quad t=T-1,\dots,1
$$

## Витерби в лог-пространстве

$$
\log \delta_t(j) =
\max_i
(
\log \delta_{t-1}(i) +
\log A_{i,j}
)
+
\log B_j(x_t)
$$

Сложение вместо умножения — стабильнее.


# 4. Где используется HMM и Витерби
🔹 POS-теггинг<br>
Слова видим, теги скрыты.

🔹 Распознавание речи<br>
Звуковой сигнал наблюдаем, фонемы скрыты.

🔹 Биология<br>
Наблюдаем нуклеотиды, скрыты состояния генома.

## Как это связано с реальным NLP

- Сейчас часто используют трансформеры, но HMM+Витерби остаются отличной базой:
  - понятная вероятностная интерпретация,
  - быстро работает,
  - помогает понимать, что делают современные sequence-модели.ели.
